<h1 style="text-align: center;">WHO Life Expectancy — Modelling Experiments</h1>

This notebook focuses on experimenting with different machine learning models for predicting `Life expectancy`. The goal is to establish baseline performance, compare model behaviour, evaluate results using appropriate regression metrics, and gradually improve the modelling workflow based on evidence from the experiments.

---
## Current Feature Engineering Decision

As of **11 September 2026**, no new engineered features have been added to the dataset. The modelling experiments will initially use the original predictor variables identified during EDA.

Data cleaning and preprocessing will still be performed where required, including handling missing values, categorical variables, and any model-specific preparation.

Additional feature engineering, feature removal, transformations, or dimensionality-reduction techniques such as PCA have not yet been decided.

These decisions will be revisited after the first modelling experiments and their results are evaluated. Any later changes to the feature set will be based on model performance, validation results, and clear analytical justification rather than being introduced in advance.

---
## Data import 

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
pd.set_option("display.float_format", "{:.2f}".format)

In [24]:
data = pd.read_csv("../Data/Data.csv")

In [25]:
data.head(3)

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.00,263.00,62,0.01,71.28,65.00,1154,...,6.00,8.16,65.00,0.10,584.26,33736494.00,17.20,17.30,0.48,10.10
1,Afghanistan,2014,Developing,59.90,271.00,64,0.01,73.52,62.00,492,...,58.00,8.18,62.00,0.10,612.70,327582.00,17.50,17.50,0.48,10.00
2,Afghanistan,2013,Developing,59.90,268.00,66,0.01,73.22,64.00,430,...,62.00,8.13,64.00,0.10,631.74,31731688.00,17.70,17.70,0.47,9.90


In [26]:
# Remove whitespace from column names to avoid inconsistent column access
data.columns = data.columns.str.strip()

In [27]:
# Remove rows with missing target values because the target should not be imputed
data.dropna(subset=['Life expectancy'], axis=0, inplace=True)

In [28]:
data.isnull().sum()

Country                              0
Year                                 0
Status                               0
Life expectancy                      0
Adult Mortality                      0
infant deaths                        0
Alcohol                            193
percentage expenditure               0
Hepatitis B                        553
Measles                              0
BMI                                 32
under-five deaths                    0
Polio                               19
Total expenditure                  226
Diphtheria                          19
HIV/AIDS                             0
GDP                                443
Population                         644
thinness  1-19 years                32
thinness 5-9 years                  32
Income composition of resources    160
Schooling                          160
dtype: int64

In [29]:
data.isnull().mean()*100

Country                            0.00
Year                               0.00
Status                             0.00
Life expectancy                    0.00
Adult Mortality                    0.00
infant deaths                      0.00
Alcohol                            6.59
percentage expenditure             0.00
Hepatitis B                       18.89
Measles                            0.00
BMI                                1.09
under-five deaths                  0.00
Polio                              0.65
Total expenditure                  7.72
Diphtheria                         0.65
HIV/AIDS                           0.00
GDP                               15.13
Population                        21.99
thinness  1-19 years               1.09
thinness 5-9 years                 1.09
Income composition of resources    5.46
Schooling                          5.46
dtype: float64

In [30]:
x = data.drop(columns=['Life expectancy'])
y = data['Life expectancy']

In [31]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.33, random_state=1)

---

## CatBoost Baseline Model

This section establishes the first regression baseline using `CatBoostRegressor`. CatBoost is used directly with the original numerical features and the categorical features `Country` and `Status`.

No scaling or manual imputation is applied in this experiment, as CatBoost can natively handle numerical missing values and categorical features. The purpose is to measure how well the original dataset performs before introducing additional preprocessing, feature engineering, or tuning.

In [46]:
from catboost import CatBoostRegressor

In [47]:
model = CatBoostRegressor(iterations=200, learning_rate=0.1, depth=6, loss_function='RMSE', random_seed=1, verbose=0, cat_features=["Country", "Status"])

In [48]:
model.fit(x_train, y_train)

CatBoostRegressor(cat_features=['Country', 'Status'], depth=6, iterations=200, learning_rate=0.1, loss_function='RMSE', random_seed=1, verbose=0)

### Evaluation

In [49]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error 

In [50]:
y_hat = model.predict(x_test)
y_hat_train = model.predict(x_train)

In [51]:
print(f"Accuracy of the model is {r2_score(y_test, y_hat)}")
print(f"training score of the model is {r2_score(y_train, y_hat_train)}")

Accuracy of the model is 0.9599718578378691
training score of the model is 0.9816404064208334


In [53]:
print(f'Mean squared error is {mean_squared_error(y_test, y_hat)}\nThe mean absolute error is {mean_absolute_error(y_test, y_hat)}')

Mean squared error is 3.50445433677003
The mean absolute error is 1.2755909127036913


### Baseline Evaluation

The initial CatBoost model achieved a strong test **R² score of approximately 0.960**, compared with a training R² of approximately **0.982**. The relatively small difference suggests no obvious severe overfitting in this initial experiment.

The model produced an **MAE of approximately 1.28 years**, meaning its predictions differ from the actual Life Expectancy by about 1.28 years on average. The MSE was approximately **3.50**.

Overall, this represents a strong first baseline. However, the validation strategy and repeated country year structure should be investigated further before treating this performance as final.

---
### Country-Based Group Split

To test whether the model can generalize to completely unseen countries, the dataset will be split using `GroupShuffleSplit` with `Country` as the grouping variable.

This ensures that each country appears entirely in either the training set or the test set, preventing country overlap between both datasets.

In [54]:
from sklearn.model_selection import GroupShuffleSplit

In [55]:
spliter= GroupShuffleSplit(n_splits=1, test_size=0.33,random_state=1)

In [56]:
train_idx, test_idx = next(spliter.split(x,y,groups=x['Country']))

In [58]:
x_train = x.iloc[train_idx]
x_test = x.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [59]:
model.fit(x_train, y_train)

CatBoostRegressor(cat_features=['Country', 'Status'], depth=6, iterations=200, learning_rate=0.1, loss_function='RMSE', random_seed=1, verbose=0)

### Evaluation

In [60]:
y_hat = model.predict(x_test)
y_hat_train = model.predict(x_train)

In [61]:
print(f"Accuracy of the model is {r2_score(y_test, y_hat)}")
print(f"training score of the model is {r2_score(y_train, y_hat_train)}")

Accuracy of the model is 0.8860535689485315
training score of the model is 0.984665543186314


In [62]:
print(f'Mean squared error is {mean_squared_error(y_test, y_hat)}\nThe mean absolute error is {mean_absolute_error(y_test, y_hat)}')

Mean squared error is 8.804677894245565
The mean absolute error is 2.105075246356713


### Country Based Split Evaluation

The country based split achieved a test R² of approximately **0.886**, compared with about **0.960** from the original random split. This shows that the model performs worse when predicting Life Expectancy for countries that were completely unseen during training.

Despite the drop, the result is still reasonably strong and shows that the model can generalize using the remaining health, economic, and demographic features.

However, this experiment represents a much harder prediction task because `Country` is an informative feature and the model has no historical information about the test countries. Therefore, this result will be treated as an important robustness check rather than the final modelling strategy.